In [1]:
import os
import sys
import subprocess
import pandas as pd
import numpy as np

def bootstrap():
    for lib in ['pandas', 'numpy']:
        try: __import__(lib)
        except ImportError: subprocess.check_call([sys.executable, "-m", "pip", "install", lib])
    for f in ['data/raw', 'data/chunks', 'models']: os.makedirs(f, exist_ok=True)

bootstrap()

# --- CONFIG ---
INPUT_FILE = 'data/raw/filtered_2024_2026.csv' # Place your big file here
TARGET_SIZE = 100000

print("📂 Loading and Sorting Dataset...")
df = pd.read_csv(INPUT_FILE)
df.columns = [c.lower() for c in df.columns]
df['acq_date'] = pd.to_datetime(df['acq_date'])
df = df.sort_values(by=['acq_date', 'acq_time'])

# Seasonal Weighting (5x for Summer)
df['month'] = df['acq_date'].dt.month
weights = df['month'].apply(lambda x: 5.0 if 2 <= x <= 6 else 1.0)

# Sample & Re-sort
sampled_indices = df.sample(n=TARGET_SIZE, weights=weights, random_state=42).index
df_sampled = df.loc[sampled_indices].sort_values(by=['acq_date', 'acq_time']).reset_index(drop=True)

# Split into 3 Parts
chunk_size = int(np.ceil(len(df_sampled) / 3))
filenames = ['part_1_early.csv', 'part_2_mid.csv', 'part_3_late.csv']

for i in range(3):
    part = df_sampled.iloc[i*chunk_size : (i+1)*chunk_size]
    part.to_csv(f'data/chunks/{filenames[i]}', index=False)
    print(f"✅ Saved {filenames[i]} ({len(part)} rows)")

print("\n🚀 ACTION: Distribute these 3 files to your teammates.")

📂 Loading and Sorting Dataset...
✅ Saved part_1_early.csv (33334 rows)
✅ Saved part_2_mid.csv (33334 rows)
✅ Saved part_3_late.csv (33332 rows)

🚀 ACTION: Distribute these 3 files to your teammates.


In [2]:
import os
import sys
import subprocess
import time
import pandas as pd
import requests

def setup():
    libs = ['pandas', 'requests', 'numpy']
    for lib in libs:
        try: __import__(lib)
        except ImportError: subprocess.check_call([sys.executable, "-m", "pip", "install", lib])

setup()

# --- 1. FILE SELECTION ---
files = [f for f in os.listdir('data/chunks') if f.endswith('.csv') and not f.startswith('enriched_')]
if not files:
    print("❌ No files found in data/chunks/"); sys.exit()

print("Available files:"); [print(f"[{i}] {f}") for i, f in enumerate(files)]
choice = int(input("Select file number: "))
IN_PATH = f"data/chunks/{files[choice]}"
OUT_PATH = f"data/chunks/enriched_{files[choice]}"

# --- 2. DATA LOAD ---
print(f"📖 Reading {IN_PATH}...")
df = pd.read_csv(IN_PATH, on_bad_lines='skip')
df['acq_date'] = pd.to_datetime(df['acq_date']).dt.strftime('%Y-%m-%d')
df['hour_idx'] = df['acq_time'].apply(lambda x: int(str(int(x)).zfill(4)[:2]) if pd.notnull(x) else 12)

# --- 3. CHECKPOINT & HEADER FIX ---
if os.path.exists(OUT_PATH):
    try:
        header = pd.read_csv(OUT_PATH, nrows=0).columns.tolist()
        if 'temp' not in header:
            print("⚠️ Fixing missing headers...")
            temp_df = pd.read_csv(OUT_PATH, on_bad_lines='skip')
            for col in ['temp', 'humidity', 'wind']: temp_df[col] = pd.NA
            temp_df.to_csv(OUT_PATH, index=False)
        
        done_df = pd.read_csv(OUT_PATH, on_bad_lines='skip')
        done_keys = set(done_df['latitude'].round(4).astype(str) + "_" + done_df['acq_date'])
        print(f"🔄 Resuming. Found {len(done_keys)} rows already done.")
    except:
        done_keys = set()
else:
    new_cols = list(df.columns) + ['temp', 'humidity', 'wind']
    pd.DataFrame(columns=new_cols).to_csv(OUT_PATH, index=False)
    done_keys = set()
    print("🆕 Created new output file.")

# --- 4. AGGRESSIVE LOOP (Batch 50 / Wait 10s) ---
print(f"🚀 Starting Aggressive Loop. Total Rows: {len(df)}")
total_saved = 0
grouped = df.groupby('acq_date')

for date_str, group in grouped:
    to_proc = group[~(group['latitude'].round(4).astype(str) + "_" + date_str).isin(done_keys)]
    if to_proc.empty: continue

    # Batch 50
    for j in range(0, len(to_proc), 75):
        batch = to_proc.iloc[j:j+75]
        params = {
            "latitude": ",".join(map(str, batch['latitude'])),
            "longitude": ",".join(map(str, batch['longitude'])),
            "start_date": date_str, "end_date": date_str,
            "hourly": "temperature_2m,relative_humidity_2m,wind_speed_10m"
        }
        
        success = False
        retries = 0
        while not success and retries < 5:
            try:
                r = requests.get("https://archive-api.open-meteo.com/v1/archive", params=params, timeout=60)
                
                if r.status_code == 200:
                    res = r.json()
                    res_list = res if isinstance(res, list) else [res]
                    enriched = []
                    for idx, (_, row) in enumerate(batch.iterrows()):
                        try:
                            w = res_list[idx]['hourly']
                            h = int(row['hour_idx'])
                            enriched.append({
                                **row.to_dict(),
                                'temp': w['temperature_2m'][h],
                                'humidity': w['relative_humidity_2m'][h],
                                'wind': w['wind_speed_10m'][h]
                            })
                        except: continue

                    if enriched:
                        pd.DataFrame(enriched).to_csv(OUT_PATH, mode='a', header=False, index=False)
                        total_saved += len(enriched)
                        print(f"💾 Saved {len(enriched)} | Total: {total_saved} | {date_str}", end='\r')
                    
                    success = True
                    time.sleep(1.5) # Fast delay

                elif r.status_code == 429:
                    print(f"\n🛑 Rate Limit. Sleeping 15s...") # <--- AGGRESSIVE WAIT
                    time.sleep(15)
                    retries += 1
                else:
                    print(f"\n⚠️ API Error {r.status_code}. Retrying...")
                    time.sleep(5)
                    retries += 1

            except Exception as e:
                print(f"\n⚠️ Network Error: Retrying in 5s...")
                time.sleep(5)
                retries += 1
        
        if retries >= 5:
            print(f"\n❌ Failed batch. Skipping.")

print(f"\n\n🎉 Done! Final File: {OUT_PATH}")

Available files:
[0] part_1_early.csv
[1] part_2_mid.csv
[2] part_3_late.csv
📖 Reading data/chunks/part_3_late.csv...
🔄 Resuming. Found 32819 rows already done.
🚀 Starting Aggressive Loop. Total Rows: 33332
💾 Saved 22 | Total: 359 | 2025-10-31

🎉 Done! Final File: data/chunks/enriched_part_3_late.csv
